# ============================================================
# HOURLY SHARP EXTRACTION FROM JSOC (MODIFIED FROM YOUR STYLE)
# ============================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

!pip install -q drms pandas tqdm

import os
import re
import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import drms

In [ ]:
# ============================================================
# 0. PATHS
# ============================================================
AR_MAIN_BASE = "/content/drive/MyDrive/AR_Stratified"
OUT_BASE = f"{AR_MAIN_BASE}/HMI_SHARP_HOURLY"
os.makedirs(OUT_BASE, exist_ok=True)

# JSOC client
c = drms.Client()

print("DRMS client ready.")
print("Output folder:", OUT_BASE)

DRMS client ready.
Output folder: /content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY


In [ ]:
# ============================================================
# 1. KEYS TO EXTRACT
# ============================================================
SHARP_KEYS = [
    "T_REC",
    "HARPNUM",
    "NOAA_ARS",
    "QUALITY",
    "MEANGBZ",
    "MEANGAM",
    "MEANGBT",
    "MEANGBH",
    "MEANJZD",
    "TOTUSJZ",
    "MEANALP",
    "MEANJZH",
    "ABSNJZH",
    "SAVNCPP",
    "MEANSHR",
    "SHRGT45",
    "R_VALUE",
    "USFLUX",
    "TOTPOT",
    "TOTUSJH",
    "AREA_ACR",
    "MEANPOT",
    "LON_MIN",
    "LON_MAX",
    "LAT_MIN",
    "LAT_MAX"
]

print("Total keys:", len(SHARP_KEYS))

Total keys: 26


In [ ]:
# ============================================================
# 2. HELPERS
# ============================================================
def extract_primary_ar(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.lower() in ["nan", "none", "missing", ""]:
        return np.nan
    m = re.search(r"\d+", s)
    return int(m.group()) if m else np.nan


def parse_trec(df):
    df = df.copy()
    df["T_REC"] = df["T_REC"].astype(str).str.strip()
    df["T_REC_dt"] = pd.to_datetime(
        df["T_REC"],
        format="%Y.%m.%d_%H:%M:%S_TAI",
        errors="coerce"
    )
    df["year"] = df["T_REC_dt"].dt.year
    return df


def clean_sharp_df(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    df = parse_trec(df)

    if "NOAA_ARS" in df.columns:
        df["NOAA_AR"] = df["NOAA_ARS"].apply(extract_primary_ar).astype("Int64")
    else:
        df["NOAA_AR"] = pd.Series([pd.NA] * len(df), dtype="Int64")

    # numeric conversion
    for col in df.columns:
        if col not in ["T_REC", "NOAA_ARS", "T_REC_dt"]:
            try:
                df[col] = pd.to_numeric(df[col], errors="ignore")
            except Exception:
                pass

    # drop bad timestamps
    df = df.dropna(subset=["T_REC_dt"]).copy()

    # quality filter if present
    if "QUALITY" in df.columns:
        df["QUALITY"] = pd.to_numeric(df["QUALITY"], errors="coerce")
        df = df[df["QUALITY"] == 0].copy()

    # deduplicate at HARPNUM + time
    subset_cols = [c for c in ["HARPNUM", "T_REC_dt"] if c in df.columns]
    if subset_cols:
        df = df.drop_duplicates(subset=subset_cols, keep="first")

    df = df.sort_values(["T_REC_dt", "HARPNUM"] if "HARPNUM" in df.columns else ["T_REC_dt"]).reset_index(drop=True)
    return df


def build_candidate_queries(start_ts, n_days):
    """
    Build a few syntax variants in case JSOC is picky.
    The important bit is:
        hmi.sharp_cea_720s[][TIME_FILTER]
    because HARPNUM is the first prime key and T_REC is the second.
    """
    s1 = start_ts.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    s2 = start_ts.strftime("%Y.%m.%d_%H:%M_TAI")

    candidates = [
        f"hmi.sharp_cea_720s[][{s1}/{n_days}d@1h]",
        f"hmi.sharp_cea_720s[][{s2}/{n_days}d@1h]",
        f"hmi.sharp_cea_720s[][{s1}/{n_days}d]",
        f"hmi.sharp_cea_720s[][{s2}/{n_days}d]",
    ]
    return candidates


def query_chunk(start_ts, n_days, keys):
    """
    Try multiple query syntaxes and return the first successful DataFrame.
    """
    candidates = build_candidate_queries(start_ts, n_days)

    last_err = None
    for ds in candidates:
        try:
            print("Trying query:", ds)
            df = c.query(ds, key=",".join(keys))
            if df is not None and len(df) > 0:
                print("Success with:", ds)
                return df, ds
        except Exception as e:
            last_err = e
            print("Failed:", e)

    raise RuntimeError(f"All query variants failed for chunk starting {start_ts}. Last error: {last_err}")

In [ ]:

# ============================================================
# 3. TEST ONE SMALL QUERY FIRST
# ============================================================
test_start = pd.Timestamp("2014-01-01 00:00:00")
test_days = 1

test_df, test_ds = query_chunk(test_start, test_days, SHARP_KEYS)
print("\nTest query worked.")
print("Query used:", test_ds)
print("Returned shape:", test_df.shape)
display(test_df.head())

Trying query: hmi.sharp_cea_720s[][2014.01.01_00:00:00_TAI/1d@1h]
Success with: hmi.sharp_cea_720s[][2014.01.01_00:00:00_TAI/1d@1h]

Test query worked.
Query used: hmi.sharp_cea_720s[][2014.01.01_00:00:00_TAI/1d@1h]
Returned shape: (235, 26)


,T_REC,HARPNUM,NOAA_ARS,QUALITY,MEANGBZ,MEANGAM,MEANGBT,MEANGBH,MEANJZD,TOTUSJZ,...,R_VALUE,USFLUX,TOTPOT,TOTUSJH,AREA_ACR,MEANPOT,LON_MIN,LON_MAX,LAT_MIN,LAT_MAX
0,2014.01.01_00:00:00_TAI,3520,"11931,11934,11935,11939",0,63.758,30.447,59.207,28.449,-0.239725,1.545143e+13,...,4.602,2.425337e+22,1.239458e+23,782.811,617.493835,2609.887,67.791039,90.513145,-22.551620,-11.495181
1,2014.01.01_01:00:00_TAI,3520,"11931,11934,11935,11939",0,63.822,29.900,58.984,27.470,-0.123619,1.432997e+13,...,4.629,2.290200e+22,1.106118e+23,722.082,587.512512,2428.895,68.348007,90.624245,-22.497293,-11.414536
2,2014.01.01_02:00:00_TAI,3520,"11931,11934,11935,11939",0,60.254,29.565,56.988,27.426,-0.178864,1.302870e+13,...,4.764,2.217570e+22,1.028660e+23,675.369,529.723206,2369.288,68.884651,90.498116,-22.410145,-11.411620
3,2014.01.01_03:00:00_TAI,3520,"11931,11934,11935,11939",0,58.173,29.099,55.478,26.581,-0.183216,1.231356e+13,...,4.695,2.158969e+22,9.661745e+22,659.351,466.931671,2317.302,69.455490,90.584892,-22.276407,-11.403701
4,2014.01.01_04:00:00_TAI,3520,"11931,11934,11935,11939",0,65.344,28.754,60.128,27.523,-0.101684,1.271466e+13,...,4.656,2.077681e+22,8.753201e+22,655.775,396.770172,2206.799,70.044754,90.724327,-22.210768,-11.431168


In [ ]:
# ============================================================
# 4. YEARLY HOURLY EXTRACTION IN 7-DAY CHUNKS
# ============================================================
def fetch_hourly_sharp_year(year, out_dir=OUT_BASE, chunk_days=7, pause_sec=1):
    start_year = pd.Timestamp(f"{year}-01-01 00:00:00")
    end_year = pd.Timestamp(f"{year+1}-01-01 00:00:00")

    print(f"\n{'='*70}")
    print(f"FETCHING YEAR {year}")
    print(f"{'='*70}")

    chunks = []
    chunk_start = start_year

    while chunk_start < end_year:
        chunk_end = min(chunk_start + pd.Timedelta(days=chunk_days), end_year)
        n_days = int((chunk_end - chunk_start).total_seconds() / 86400)

        # safety for final small chunk
        if n_days < 1:
            break

        try:
            df_chunk, ds_used = query_chunk(chunk_start, n_days, SHARP_KEYS)

            if df_chunk is not None and len(df_chunk) > 0:
                df_chunk = clean_sharp_df(df_chunk)
                chunks.append(df_chunk)
                print(f"Chunk saved in memory: {chunk_start} -> rows={len(df_chunk)}")
            else:
                print(f"No rows for chunk starting {chunk_start}")

        except Exception as e:
            print(f"Chunk failed for {chunk_start}: {e}")

        chunk_start = chunk_end
        time.sleep(pause_sec)

    if len(chunks) == 0:
        print(f"No data collected for year {year}")
        return None

    year_df = pd.concat(chunks, ignore_index=True)

    # keep only target year after parsing
    year_df = year_df[year_df["year"] == year].copy()

    # deduplicate again after concat
    dedup_cols = [c for c in ["HARPNUM", "T_REC_dt"] if c in year_df.columns]
    if dedup_cols:
        year_df = year_df.drop_duplicates(subset=dedup_cols, keep="first")

    year_df = year_df.sort_values(["HARPNUM", "T_REC_dt"]).reset_index(drop=True)

    out_path = os.path.join(out_dir, f"hmi_hourly_sharp_{year}.csv")
    year_df.to_csv(out_path, index=False)

    print(f"\nSaved year {year} to:")
    print(out_path)
    print("Final shape:", year_df.shape)

    return year_df

In [ ]:
df_2014 = fetch_hourly_sharp_year(2014, chunk_days=7)

if df_2014 is not None:
    print(df_2014.head())
    print("\nColumns:")
    print(df_2014.columns.tolist())

    print("\nGrouped cadence check within HARPNUM:")
    print(
        df_2014.sort_values(["HARPNUM", "T_REC_dt"])
               .groupby("HARPNUM")["T_REC_dt"]
               .diff()
               .value_counts()
               .head(20)
    )


FETCHING YEAR 2014
Trying query: hmi.sharp_cea_720s[][2014.01.01_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2014.01.01_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2014-01-01 00:00:00 -> rows=1840


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Trying query: hmi.sharp_cea_720s[][2014.01.08_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2014.01.08_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2014-01-08 00:00:00 -> rows=2455


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Trying query: hmi.sharp_cea_720s[][2014.01.15_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2014.01.15_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2014-01-15 00:00:00 -> rows=1454


...

Trying query: hmi.sharp_cea_720s[][2014.12.24_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2014.12.24_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2014-12-24 00:00:00 -> rows=2622


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Trying query: hmi.sharp_cea_720s[][2014.12.31_00:00:00_TAI/1d@1h]
Success with: hmi.sharp_cea_720s[][2014.12.31_00:00:00_TAI/1d@1h]
Chunk saved in memory: 2014-12-31 00:00:00 -> rows=288


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")



Saved year 2014 to:
/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/hmi_hourly_sharp_2014.csv
Final shape: (88063, 29)
                     T_REC  HARPNUM                 NOAA_ARS  QUALITY  \
0  2014.01.01_00:00:00_TAI     3520  11931,11934,11935,11939        0   
1  2014.01.01_01:00:00_TAI     3520  11931,11934,11935,11939        0   
2  2014.01.01_02:00:00_TAI     3520  11931,11934,11935,11939        0   
3  2014.01.01_03:00:00_TAI     3520  11931,11934,11935,11939        0   
4  2014.01.01_04:00:00_TAI     3520  11931,11934,11935,11939        0   

   MEANGBZ  MEANGAM  MEANGBT  MEANGBH   MEANJZD       TOTUSJZ  ...  TOTUSJH  \
0   63.758   30.447   59.207   28.449 -0.239725  1.545143e+13  ...  782.811   
1   63.822   29.900   58.984   27.470 -0.123619  1.432997e+13  ...  722.082   
2   60.254   29.565   56.988   27.426 -0.178864  1.302870e+13  ...  675.369   
3   58.173   29.099   55.478   26.581 -0.183216  1.231356e+13  ...  659.351   
4   65.344   28.754   60.128   27.523 -0

In [ ]:
all_years = list(range(2010, 2026))
year_shapes = {}

for yr in tqdm(all_years, desc="Extracting hourly SHARP by year"):
    try:
        df_year = fetch_hourly_sharp_year(yr, chunk_days=7)
        year_shapes[yr] = None if df_year is None else df_year.shape
    except Exception as e:
        print(f"Year {yr} failed: {e}")
        year_shapes[yr] = "FAILED"

print("\nDONE.")
print(year_shapes)

Extracting hourly SHARP by year:   0%|          | 0/16 [00:00<?, ?it/s]


FETCHING YEAR 2010
Trying query: hmi.sharp_cea_720s[][2010.01.01_00:00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.01_00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.01_00:00:00_TAI/7d]
Trying query: hmi.sharp_cea_720s[][2010.01.01_00:00_TAI/7d]
Chunk failed for 2010-01-01 00:00:00: All query variants failed for chunk starting 2010-01-01 00:00:00. Last error: None
Trying query: hmi.sharp_cea_720s[][2010.01.08_00:00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.08_00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.08_00:00:00_TAI/7d]
Trying query: hmi.sharp_cea_720s[][2010.01.08_00:00_TAI/7d]
Chunk failed for 2010-01-08 00:00:00: All query variants failed for chunk starting 2010-01-08 00:00:00. Last error: None
Trying query: hmi.sharp_cea_720s[][2010.01.15_00:00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.15_00:00_TAI/7d@1h]
Trying query: hmi.sharp_cea_720s[][2010.01.15_00:00:00_TAI/7d]
Trying query: hmi.sharp_cea_720s[][2010.01

/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Trying query: hmi.sharp_cea_720s[][2010.05.07_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2010.05.07_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2010-05-07 00:00:00 -> rows=384


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


...

Trying query: hmi.sharp_cea_720s[][2025.12.24_00:00:00_TAI/7d@1h]
Success with: hmi.sharp_cea_720s[][2025.12.24_00:00:00_TAI/7d@1h]
Chunk saved in memory: 2025-12-24 00:00:00 -> rows=1457


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Trying query: hmi.sharp_cea_720s[][2025.12.31_00:00:00_TAI/1d@1h]
Success with: hmi.sharp_cea_720s[][2025.12.31_00:00:00_TAI/1d@1h]
Chunk saved in memory: 2025-12-31 00:00:00 -> rows=265


/tmp/ipykernel_6771/316768405.py:41: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")



Saved year 2025 to:
/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/hmi_hourly_sharp_2025.csv
Final shape: (100897, 29)

DONE.
{2010: (21452, 29), 2011: (62145, 29), 2012: (68486, 29), 2013: (81875, 29), 2014: (88063, 29), 2015: (75981, 29), 2016: (16388, 29), 2017: (15558, 29), 2018: (7866, 29), 2019: (4917, 29), 2020: (9883, 29), 2021: (30619, 29), 2022: (70986, 29), 2023: (103314, 29), 2024: (100009, 29), 2025: (100897, 29)}


# **Cell A — merge all hourly yearly files**

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob(f"{OUT_BASE}/hmi_hourly_sharp_*.csv"))
print("Files found:", len(files))

dfs = []
for f in files:
    d = pd.read_csv(f)
    dfs.append(d)

hourly_full = pd.concat(dfs, ignore_index=True)
hourly_full["T_REC_dt"] = pd.to_datetime(hourly_full["T_REC_dt"], errors="coerce")
hourly_full["HARPNUM"] = pd.to_numeric(hourly_full["HARPNUM"], errors="coerce").astype("Int64")
hourly_full["NOAA_AR"] = pd.to_numeric(hourly_full["NOAA_AR"], errors="coerce").astype("Int64")

hourly_full = hourly_full.dropna(subset=["T_REC_dt", "HARPNUM"]).copy()
hourly_full = hourly_full.sort_values(["HARPNUM", "T_REC_dt"]).reset_index(drop=True)

merged_out = f"{OUT_BASE}/HMI_SHARP_HOURLY_2010_2025_MERGED.csv"
hourly_full.to_csv(merged_out, index=False)

print("Merged shape:", hourly_full.shape)
print("Saved merged file to:")
print(merged_out)

Files found: 16
Merged shape: (858439, 29)
Saved merged file to:
/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/HMI_SHARP_HOURLY_2010_2025_MERGED.csv


# **Cell B — Then build your ML-ready hourly 16-feature file**

In [ ]:
core_features = [
    "T_REC_dt", "year", "NOAA_AR",
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH",
    "MEANJZD", "TOTUSJZ", "MEANALP", "MEANJZH",
    "ABSNJZH", "SAVNCPP", "MEANSHR", "SHRGT45",
    "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH"
]

available_core = [c for c in core_features if c in hourly_full.columns]
missing_core = [c for c in core_features if c not in hourly_full.columns]

print("Available core:", available_core)
print("Missing core:", missing_core)

ml16_hourly = hourly_full[available_core].copy()
ml16_hourly = ml16_hourly.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

ml16_out = f"{OUT_BASE}/HMI_AR_2010_2025_ML_READY_16_HOURLY.csv"
ml16_hourly.to_csv(ml16_out, index=False)

print("Saved hourly ML16 dataset to:")
print(ml16_out)
print("Shape:", ml16_hourly.shape)

Available core: ['T_REC_dt', 'year', 'NOAA_AR', 'MEANGBZ', 'MEANGAM', 'MEANGBT', 'MEANGBH', 'MEANJZD', 'TOTUSJZ', 'MEANALP', 'MEANJZH', 'ABSNJZH', 'SAVNCPP', 'MEANSHR', 'SHRGT45', 'R_VALUE', 'USFLUX', 'TOTPOT', 'TOTUSJH']
Missing core: []
Saved hourly ML16 dataset to:
/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/HMI_AR_2010_2025_ML_READY_16_HOURLY.csv
Shape: (858439, 19)


# **Cell C — parse GOES and prepare M/X + all-flare tables**

In [ ]:
import glob
import re
import numpy as np
import pandas as pd

AR_MAIN_BASE = "/content/drive/MyDrive/AR_Stratified"
GOES_BASE = f"{AR_MAIN_BASE}/GOES_XRAY_EVENTS"

goes_files = sorted(glob.glob(f"{GOES_BASE}/GOES XRAY events for *.txt"))
print("GOES files found:", len(goes_files))

def parse_goes_line(line):
    line = line.strip()
    if not line or "Written" in line or "GOES" in line or "events" in line.lower():
        return None

    tokens = line.split()
    if len(tokens) < 4:
        return None

    if not re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$", tokens[0]):
        return None

    date = tokens[0]

    class_idx = None
    for i, t in enumerate(tokens):
        if re.match(r"^[A-Z][0-9](\.[0-9])?$", t):
            class_idx = i
            break

    if class_idx is None:
        return None

    start_time = tokens[1] if len(tokens) > 1 else None
    peak_time = tokens[2] if len(tokens) > 2 else None
    flare_class = tokens[class_idx]
    location = tokens[class_idx + 1] if class_idx + 1 < len(tokens) else None

    noaa_ar = np.nan
    for t in tokens[class_idx + 1:]:
        if re.fullmatch(r"\d{4,5}", t):
            noaa_ar = int(t)
            break

    return {
        "flare_date": date,
        "start_time": start_time,
        "peak_time": peak_time,
        "flare_class": flare_class,
        "location": location,
        "NOAA_AR": noaa_ar,
    }

rows = []
for file in goes_files:
    with open(file, "r", errors="ignore") as f:
        for line in f:
            parsed = parse_goes_line(line)
            if parsed is not None:
                rows.append(parsed)

goes_df = pd.DataFrame(rows)
print("Raw parsed GOES shape:", goes_df.shape)

goes_df = goes_df.dropna(subset=["flare_date", "start_time"]).copy()
goes_df["flare_start_dt"] = pd.to_datetime(
    goes_df["flare_date"] + " " + goes_df["start_time"],
    errors="coerce"
)
goes_df = goes_df.dropna(subset=["flare_start_dt"]).copy()
goes_df["NOAA_AR"] = pd.to_numeric(goes_df["NOAA_AR"], errors="coerce").astype("Int64")
goes_df["flare_class"] = goes_df["flare_class"].astype(str).str.strip()
goes_df["class_letter"] = goes_df["flare_class"].str[0]
goes_df["class_mag"] = pd.to_numeric(goes_df["flare_class"].str[1:], errors="coerce")
goes_df = goes_df.dropna(subset=["NOAA_AR", "class_letter"]).copy()

def get_flux_value(flare_class_str):
    if pd.isna(flare_class_str):
        return np.nan
    s = str(flare_class_str).strip()
    if len(s) < 2:
        return np.nan

    letter = s[0].upper()
    try:
        mag = float(s[1:])
    except ValueError:
        return np.nan

    base_map = {
        "A": 1e-8,
        "B": 1e-7,
        "C": 1e-6,
        "M": 1e-5,
        "X": 1e-4,
    }
    if letter not in base_map:
        return np.nan
    return mag * base_map[letter]

goes_df["peak_flux"] = goes_df["flare_class"].apply(get_flux_value)
goes_df["year"] = goes_df["flare_start_dt"].dt.year

goes_all = goes_df.sort_values(["NOAA_AR", "flare_start_dt"]).reset_index(drop=True)
goes_mx = goes_all[goes_all["class_letter"].isin(["M", "X"])].copy()
goes_mx = goes_mx.sort_values(["NOAA_AR", "flare_start_dt"]).reset_index(drop=True)

print("GOES all shape:", goes_all.shape)
print("GOES M/X shape:", goes_mx.shape)
display(goes_mx.head())

GOES files found: 16
Raw parsed GOES shape: (34723, 6)
GOES all shape: (27667, 11)
GOES M/X shape: (2240, 11)


,flare_date,start_time,peak_time,flare_class,location,NOAA_AR,flare_start_dt,class_letter,class_mag,peak_flux,year
0,20-Jan-2010,10:46,10:59,M1.8,S25E87,11041,2010-01-20 10:46:00,M,1.8,0.000018,2010
1,7-Feb-2010,02:20,02:34,M6.4,N21E11,11045,2010-02-07 02:20:00,M,6.4,0.000064,2010
2,8-Feb-2010,11:57,12:03,M1.1,N21W07,11045,2010-02-08 11:57:00,M,1.1,0.000011,2010
3,12-Feb-2010,11:19,11:26,M8.3,N33E13,11046,2010-02-12 11:19:00,M,8.3,0.000083,2010
4,12-Feb-2010,17:52,18:08,M1.1,N33E10,11046,2010-02-12 17:52:00,M,1.1,0.000011,2010


# **Cell D — build hourly 3H features and labels**

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

df3h = ml16_hourly.copy()

df3h["flare_count_past_3h"] = 0
df3h["time_since_last_flare_hours_3h"] = np.nan
df3h["max_peak_flux_past_3h"] = np.nan
df3h["mean_peak_flux_past_3h"] = np.nan
df3h["flare_activity_index_past_3h"] = 0.0
df3h["label_MX_3h"] = 0
df3h["future_flare_start_dt_mx3h"] = pd.NaT
df3h["had_flare_past_3h"] = 0

df3h = df3h.sort_values(["NOAA_AR", "T_REC_dt"]).reset_index(drop=True)

unique_ars = df3h["NOAA_AR"].dropna().unique()

print("Building 3H flare-history features...")
for ar_num in tqdm(unique_ars, desc="Past 3H features"):
    ar_idx = df3h.index[df3h["NOAA_AR"] == ar_num]
    goes_subset = goes_all[goes_all["NOAA_AR"] == ar_num].copy()

    if goes_subset.empty:
        continue

    flare_times = goes_subset["flare_start_dt"].values
    flare_flux = goes_subset["peak_flux"].values

    for idx in ar_idx:
        current_time = df3h.at[idx, "T_REC_dt"]
        window_start = current_time - pd.Timedelta(hours=3)

        # strictly past only: (t-3h, t)
        mask_past = (
            (flare_times > np.datetime64(window_start)) &
            (flare_times < np.datetime64(current_time))
        )

        relevant_flux = flare_flux[mask_past]
        relevant_times = flare_times[mask_past]

        df3h.at[idx, "flare_count_past_3h"] = int(mask_past.sum())

        if mask_past.sum() > 0:
            df3h.at[idx, "max_peak_flux_past_3h"] = np.nanmax(relevant_flux)
            df3h.at[idx, "mean_peak_flux_past_3h"] = np.nanmean(relevant_flux)
            df3h.at[idx, "flare_activity_index_past_3h"] = np.nansum(relevant_flux)
            last_flare_time = pd.Timestamp(relevant_times.max())
            delta_hours = (current_time - last_flare_time).total_seconds() / 3600.0
            df3h.at[idx, "time_since_last_flare_hours_3h"] = delta_hours
            df3h.at[idx, "had_flare_past_3h"] = 1

print("Applying M/X 3H future labels...")
for _, row in tqdm(goes_mx.iterrows(), total=len(goes_mx), desc="MX 3H labels"):
    ar_num = row["NOAA_AR"]
    flare_time = row["flare_start_dt"]

    start_win = flare_time - pd.Timedelta(hours=3)
    end_win = flare_time

    mask = (
        (df3h["NOAA_AR"] == ar_num) &
        (df3h["T_REC_dt"] >= start_win) &
        (df3h["T_REC_dt"] < end_win)
    )

    if mask.any():
        df3h.loc[mask, "label_MX_3h"] = 1
        df3h.loc[mask, "future_flare_start_dt_mx3h"] = flare_time

print("Done.")

Building 3H flare-history features...


Past 3H features:   0%|          | 0/2305 [00:00<?, ?it/s]

Applying M/X 3H future labels...


MX 3H labels:   0%|          | 0/2240 [00:00<?, ?it/s]

Done.


# **Cell E — save and verify the final 3H dataset**

In [ ]:
OUT_BASE = "/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY"
out_3h = f"{OUT_BASE}/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3H_FLARE_FEATURES_HOURLY.csv"

df3h.to_csv(out_3h, index=False)

print("Saved 3H hourly dataset to:")
print(out_3h)
print("Shape:", df3h.shape)

print("\nLabel distribution:")
print(df3h["label_MX_3h"].value_counts(dropna=False))

print("\nPositive ratio:")
print(df3h["label_MX_3h"].value_counts(normalize=True, dropna=False))

print("\nGrouped cadence check within NOAA_AR:")
print(
    df3h.sort_values(["NOAA_AR", "T_REC_dt"])
       .groupby("NOAA_AR")["T_REC_dt"]
       .diff()
       .value_counts()
       .head(20)
)

print("\nLeakage check:")
tmp = df3h.copy()
tmp["future_flare_start_dt_mx3h"] = pd.to_datetime(tmp["future_flare_start_dt_mx3h"], errors="coerce")
leak_rows = tmp[
    (tmp["label_MX_3h"] == 1) &
    tmp["future_flare_start_dt_mx3h"].notna() &
    (tmp["future_flare_start_dt_mx3h"] <= tmp["T_REC_dt"])
]
print("Future-label leakage rows:", len(leak_rows))

display(
    df3h[
        [
            "T_REC_dt", "NOAA_AR", "label_MX_3h",
            "flare_count_past_3h", "had_flare_past_3h",
            "time_since_last_flare_hours_3h",
            "max_peak_flux_past_3h",
            "mean_peak_flux_past_3h",
            "flare_activity_index_past_3h",
            "future_flare_start_dt_mx3h"
        ]
    ].head(20)
)

Saved 3H hourly dataset to:
/content/drive/MyDrive/AR_Stratified/HMI_SHARP_HOURLY/HMI_AR_2010_2025_ML_READY_16_LABELED_MX3H_FLARE_FEATURES_HOURLY.csv
Shape: (858439, 27)

Label distribution:
label_MX_3h
0    855388
1      3051
Name: count, dtype: int64

Positive ratio:
label_MX_3h
0    0.996446
1    0.003554
Name: proportion, dtype: float64

Grouped cadence check within NOAA_AR:
T_REC_dt
0 days 01:00:00    419598
0 days 02:00:00     43189
0 days 00:00:00      7271
0 days 03:00:00      5191
0 days 05:00:00      2381
0 days 06:00:00       574
0 days 04:00:00       468
0 days 08:00:00       385
0 days 07:00:00       283
0 days 09:00:00       121
1 days 01:00:00        62
0 days 10:00:00        58
0 days 15:00:00        36
0 days 11:00:00        34
0 days 14:00:00        28
0 days 18:00:00        25
0 days 12:00:00        25
0 days 17:00:00        16
0 days 13:00:00        15
0 days 23:00:00         9
Name: count, dtype: int64

Leakage check:
Future-label leakage rows: 0


,T_REC_dt,NOAA_AR,label_MX_3h,flare_count_past_3h,had_flare_past_3h,time_since_last_flare_hours_3h,max_peak_flux_past_3h,mean_peak_flux_past_3h,flare_activity_index_past_3h,future_flare_start_dt_mx3h
0,2010-05-03 05:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
1,2010-05-03 07:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
2,2010-05-03 08:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
3,2010-05-03 09:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
4,2010-05-03 10:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
5,2010-05-03 11:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
6,2010-05-03 13:00:00,11063,0,0,0,NaN,NaN,NaN,0.000000e+00,NaT
7,2010-05-03 14:00:00,11063,0,1,1,0.366667,1.500000e-07,1.500000e-07,1.500000e-07,NaT
8,2010-05-03 15:00:00,11063,0,1,1,1.366667,1.500000e-07,1.500000e-07,1.500000e-07,NaT
9,2010-05-03 16:00:00,11063,0,1,1,2.366667,1.500000e-07,1.500000e-07,1.500000e-07,NaT


In [ ]:
print(df3h.shape)
print(df3h["label_MX_3h"].value_counts())
print(df3h["label_MX_3h"].value_counts(normalize=True))
print(
    df3h.sort_values(["NOAA_AR", "T_REC_dt"])
       .groupby("NOAA_AR")["T_REC_dt"]
       .diff()
       .value_counts()
       .head(20)
)

(858439, 27)
label_MX_3h
0    855388
1      3051
Name: count, dtype: int64
label_MX_3h
0    0.996446
1    0.003554
Name: proportion, dtype: float64
T_REC_dt
0 days 01:00:00    419598
0 days 02:00:00     43189
0 days 00:00:00      7271
0 days 03:00:00      5191
0 days 05:00:00      2381
0 days 06:00:00       574
0 days 04:00:00       468
0 days 08:00:00       385
0 days 07:00:00       283
0 days 09:00:00       121
1 days 01:00:00        62
0 days 10:00:00        58
0 days 15:00:00        36
0 days 11:00:00        34
0 days 14:00:00        28
0 days 18:00:00        25
0 days 12:00:00        25
0 days 17:00:00        16
0 days 13:00:00        15
0 days 23:00:00         9
Name: count, dtype: int64
